In [ ]:
# https://github.com/huggingface/sentence-transformers/blob/8024980cf0c4b4bb977041cafd2aa90ef7b3c560/sentence_transformers/sparse_encoder/model.py#L35
# https://github.com/huggingface/sentence-transformers/blob/328398d22050ac64737f2400c9283041b08aa59a/sentence_transformers/sparse_encoder/modules/splade_pooling.py#L14
# https://github.com/huggingface/sentence-transformers/blob/328398d22050ac64737f2400c9283041b08aa59a/sentence_transformers/base/model.py#L47

In [9]:
import torch 

torch.tensor([
  [0.0, 0.0, 1.7, 0.0, 0.0, 2.3],
  [0.0, 5.1, 0.0, 0.0, 0.0, 0.0]
]).to_sparse()

tensor(indices=tensor([[0, 0, 1],
                       [2, 5, 1]]),
       values=tensor([1.7000, 2.3000, 5.1000]),
       size=(2, 6), nnz=3, layout=torch.sparse_coo)

In [7]:
import torch

dense = torch.tensor([
  [0.0, 0.0, 1.7, 0.0, 0.0, 2.3],
  [0.0, 5.1, 0.0, 0.0, 0.0, 0.0]
])
sp = dense.to_sparse()

print(sp.indices())      # row/col of each nonzero
print(sp.values())       # the nonzero values


tensor([[0, 0, 1],
        [2, 5, 1]])
tensor([1.7000, 2.3000, 5.1000])


In [ ]:
import torch
from sentence_transformers.sparse_encoder.modules import SpladePooling
from sentence_transformers.base.modality_types import TextInput

# TextInput: str

pool = SpladePooling(pooling_strategy="max", activation_function="log1p_relu")
pool.eval()  # important: inference mode

# token_embeddings has shape:

# (1, 3, 4)

# meaning:

# 1 sentence in the batch
# 3 token positions
# for each token position, 4 vocabulary scores
features = {
    "token_embeddings": torch.tensor([
        [  # batch item 0
            [ 1.0, -1.0,  0.0, 1.0],   # token 1 logits
            [ 0.5,  3.0, -2.0, 3.0],   # token 2 logits
            [10.0, 10.0, 10.0, 10.0],  # token 3 logits, but masked out below
        ]
    ]),
    "attention_mask": torch.tensor([
        [1, 1, 0] # marks token 3 as padding
    ])
}

out = pool(features)
print(out["sentence_embedding"])

tensor([[0.6931, 1.3863, 0.0000, 1.3863]])


In [13]:
print(torch.expm1(out["sentence_embedding"]))


tensor([[1., 3., 0., 3.]])


Why this creates sparse vectors

Because:

* negative logits are killed by **ReLU**
* masked tokens are zeroed out
* many vocab terms never get positive evidence
* so many dimensions stay exactly **0**

## Back to the real model

`embeddings["sentence_embedding"]` has one dimension per vocabulary entry. Let's check its shape and how many dimensions are actually non-zero.

In [1]:
from sentence_transformers import SparseEncoder
from sentence_transformers.sparse_encoder.modules import MLMTransformer, SpladePooling

mlm = MLMTransformer("nickprock/splade-bert-base-italian-xxl-uncased-cv")
pooling = SpladePooling(pooling_strategy="max", activation_function="relu")

model = SparseEncoder(modules=[mlm, pooling], device="cpu")
tok = model.tokenizer


/home/lpozzi/Git/data-science-lectures/ir/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The MLMTransformer module is deprecated. Please use the Transformer module with `transformer_task="fill-mask"` instead.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3410.83it/s]


In [2]:
texts = ['hello words', 'hello worlds']
features = model.preprocess(texts)
embeddings = model(features)

In [3]:
emb = embeddings["sentence_embedding"].detach()

print("shape:", emb.shape)                       # (batch, vocab_size)
print("active dims per text:", (emb > 0).sum(dim=1).tolist())
print("sparsity:", 1 - (emb > 0).float().mean(dim=1))

shape: torch.Size([2, 32102])
active dims per text: [15, 6]
sparsity: tensor([0.9995, 0.9998])


## Which input token produced each term?

Max pooling throws away the token that "won" each vocabulary dimension, so it cannot be recovered from the sentence vector alone. We can recompute it from `token_embeddings`: apply the same mask, ReLU and `log1p`, then take `argmax` over the token axis instead of `max`.

When the winning input token is *different* from the term itself, that is **term expansion**: the model activated a related word that was not in the text.

In [4]:
import torch

token_logits = embeddings["token_embeddings"].detach()      # (B, T, V)
mask = embeddings["attention_mask"].unsqueeze(-1)            # (B, T, 1)

# same transformation SpladePooling applies, but we keep the token dimension
acts = torch.log1p(torch.relu(token_logits * mask))          # (B, T, V)
_, winner = acts.max(dim=1)                                  # (B, V): index of the winning token

input_tokens = [tok.convert_ids_to_tokens(ids) for ids in embeddings["input_ids"]]

for b, text in enumerate(texts):
    print(f"\n{text!r}  ->  {input_tokens[b]}")
    vals, ids = emb[b].topk(10)
    for v, i in zip(vals, ids):
        t = winner[b, i].item()
        print(f"  {tok.convert_ids_to_tokens(i.item()):>15}  {v.item():.3f}   from input token {t} ({input_tokens[b][t]})")


'hello words'  ->  ['[CLS]', 'hell', '##o', 'word', '##s', '[SEP]']
       javascript  0.583   from input token 5 ([SEP])
              pdf  0.477   from input token 0 ([CLS])
             bill  0.302   from input token 5 ([SEP])
            login  0.297   from input token 5 ([SEP])
              xxx  0.230   from input token 0 ([CLS])
           cookie  0.225   from input token 5 ([SEP])
             word  0.174   from input token 3 (word)
               no  0.165   from input token 0 ([CLS])
             file  0.137   from input token 0 ([CLS])
             jack  0.135   from input token 0 ([CLS])

'hello worlds'  ->  ['[CLS]', 'hell', '##o', 'world', '##s', '[SEP]']
              ##o  0.143   from input token 2 (##o)
              and  0.105   from input token 0 ([CLS])
             jack  0.080   from input token 0 ([CLS])
              xxx  0.063   from input token 2 (##o)
               no  0.050   from input token 0 ([CLS])
             bill  0.029   from input token 5 ([SEP])
 